In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import json
import pathlib
from functools import partial

from itertools import combinations

from copy import deepcopy
from itertools import combinations
import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 9})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
use_f64 = True

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", use_f64)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, 
    load_excitation_result, list_excitation_results, provide_excitation_results_overview,
    load_testing_sequences,
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts
from admpe.sets import SafelyExcitableSet, SafeStateSet

from admpe.excitation.excitation_experiment_setup import (
    setup_environment,
    construct_target_distribution,
)
from dmpe.utils.density_estimation import build_grid, DensityEstimate
from dmpe.utils.metrics import JSDLoss

# TODO:
<!-- 
- show excitation curves
- show constraint violation curves? -->
- show in-between data scores
- show target to target scores
- show data to target scores
- show training data to testing data scores

In [ ]:
EXPERIMENT = "submission_nov"
EXCITATION_MODEL = "perfect"

OBJECTIVES = (
    "dmpe_uniform",
    "dmpe_static_dynamics",
    "dmpe_static_mixture_dynamics",
    "dmpe_static_curv",
    "dmpe_static_mixture_curv",
    # "dmpe_static_LOLA_ls",
)

SYSTEMS = [
    "fluid_tank",
    "pendulum",
    "cart_pole",
    "pmsm_nominal",
]

In [ ]:
analysis = {system: {} for system in SYSTEMS}

In [ ]:
# list all SE runs for the specified setup

for system in SYSTEMS:
    entries = []

    for objective in OBJECTIVES:
        source = EXPERIMENT
    
        if system == "pmsm_nominal" and objective in (
            "dmpe_static_curv",
            "dmpe_static_mixture_curv",
        ):
            source = "submission_nov_curv64"  # EXPERIMENT
        
        entries.extend(
            list_excitation_results(
                system_name=system,
                exp_name=source,
                model_type=EXCITATION_MODEL,
                excitation_objective=objective,
                status="completed",
            )
        )
    
    if not entries:
        raise ValueError(f"No completed runs found for {system}.")
    
    overview = provide_excitation_results_overview(entries)
    seed_counts = pd.crosstab(
        overview["objective"],
        overview["seed"],
    ).reindex(index=OBJECTIVES, fill_value=0)
    
    print(f"System: {system}")
    
    if not seed_counts.eq(1).all().all():
        raise ValueError(
            "Expected exactly one dataset for every objective–seed combination."
        )
    
    if len(seed_counts.columns) < 2:
        raise ValueError(
            "At least two seeds are needed to compare different runs "
            "of the same objective."
        )
    analysis[system].update({
        "entries": entries,
        "overview": overview,
        "seed_counts": seed_counts,
    })

    print(f"System: {system} | Excitation datasets: {len(entries)}")
    display(seed_counts)

In [ ]:
# check same grid shape for data density estimate

for system in SYSTEMS:

    system_data = analysis[system]
    runs = {}

    
    for entry in system_data["entries"]:
        result = load_excitation_result(entry["path"], dtype=jnp.float32 if not use_f64 else jnp.float64)
        metadata = result["metadata"]
        key = (metadata["excitation_objective"], int(metadata["seed"]))
        runs[key] = result
    
    reference = next(iter(runs.values()))
    reference_estimate = reference["estimator"]
    reference_metadata = reference["metadata"]
    
    grid = np.asarray(reference_estimate.z_g)
    bandwidth = float(np.asarray(reference_estimate.bandwidth).item())
    use_actions = reference_metadata["params"]["alg_params"]["consider_action_distribution"]
    
    checks = []
    
    for (objective, seed), result in runs.items():
        estimate = result["estimator"]
        metadata = result["metadata"]
        alg = metadata["params"]["alg_params"]
    
        # Grid coordinates and their ordering must agree.
        np.testing.assert_allclose(
            estimate.z_g, grid, rtol=0, atol=1e-7,
        )
        np.testing.assert_allclose(
            estimate.bandwidth, reference_estimate.bandwidth,
        )
    
        if alg["consider_action_distribution"] != use_actions:
            raise ValueError("Runs use different state/action feature definitions.")
    
        if metadata["params"]["env_params"] != reference_metadata["params"]["env_params"]:
            raise ValueError("Runs use different environment configurations.")
    
        values = np.asarray(estimate.p).reshape(-1)
        if (
            len(values) != len(grid)
            or not np.isfinite(values).all()
            or (values < 0).any()
            or values.sum() <= 0
        ):
            raise ValueError(f"Invalid density estimate: {objective}, seed {seed}")
    
        if not all(
            np.isfinite(result[name]).all()
            for name in ("observations", "actions")
        ):
            raise ValueError(f"Nonfinite trajectory: {objective}, seed {seed}")
    
        checks.append({
            "objective": objective,
            "seed": seed,
            "n_observations": int(np.asarray(estimate.n_observations).item()),
        })

    checks = pd.DataFrame(checks)
    system_data.update({
        "runs": runs,
        "reference": reference,
        "reference_estimate": reference_estimate,
        "reference_metadata": reference_metadata,
        "grid": grid,
        "bandwidth": bandwidth,
        "use_actions": use_actions,
        "obs_dim": reference["observations"].shape[-1],
        "density_checks": checks,
    })
    
    print(f"Distribution features: {'states and actions' if use_actions else 'states only'}")
    print(f"Feature dimension: {grid.shape[1]}")
    print(f"Grid points: {len(grid)}")
    print(f"KDE bandwidth: {bandwidth:.4g}")
    
    display(
        checks.groupby("objective")
        .agg(
            datasets=("seed", "size"),
            observations_min=("n_observations", "min"),
            observations_max=("n_observations", "max"),
        )
        .reindex(OBJECTIVES)
    )

In [ ]:
def normalize(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)

    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Probability values must be finite and nonnegative.")

    mass = values.sum()
    if not np.isfinite(mass) or mass <= 0:
        raise ValueError("Distribution has no finite positive mass.")

    return values / mass


def restrict(values, safe_mask):
    full_pmf = normalize(values)

    if full_pmf.shape != safe_mask.shape:
        raise ValueError("Distribution/grid shape mismatch.")

    return normalize(full_pmf[safe_mask])


for system in SYSTEMS:
    system_data = analysis[system]
    grid = system_data["grid"]
    runs = system_data["runs"]

    ses, set_path = SafelyExcitableSet.from_system_name(system)

    if system_data["use_actions"]:
        domain = ses
    else:
        domain = SafeStateSet.from_safely_excitable_set(
            ses, system_data["obs_dim"],
        )

    if domain.dim != grid.shape[1]:
        raise ValueError(
            f"{system}: safe domain and density grid have different dimensions."
        )

    safe_mask = np.asarray(domain.contains(grid), dtype=bool)
    safe_grid = grid[safe_mask]

    if len(safe_grid) == 0:
        raise ValueError(f"{system}: no grid points lie inside the safe domain.")

    densities = {}
    restriction_rows = []

    for (objective, seed), result in runs.items():
        values = result["estimator"].p

        densities[(objective, seed)] = restrict(values, safe_mask)

        restriction_rows.append({
            "objective": objective,
            "seed": seed,
            "retained_grid_mass": normalize(values)[safe_mask].sum(),
        })

    restriction_summary = pd.DataFrame(restriction_rows)

    system_data.update({
        "ses": ses,
        "set_path": set_path,
        "domain": domain,
        "safe_mask": safe_mask,
        "safe_grid": safe_grid,
        "densities": densities,
        "restriction_summary": restriction_summary,
    })

    print(f"System: {system}")
    print(f"Safe-set file: {set_path.name}")
    print(f"Retained grid points: {len(safe_grid)} / {len(grid)}")

    display(
        restriction_summary.groupby("objective")["retained_grid_mass"]
        .agg(["mean", "min", "max"])
        .reindex(OBJECTIVES)
    )

In [ ]:
analysis.keys()

In [ ]:
for system in SYSTEMS:
    system_data = analysis[system]
    targets = {}

    for objective in OBJECTIVES:
        saved_targets = [
            result["metadata"]["params"]["alg_params"]["target_distribution"]
            for (run_objective, seed), result
            in system_data["runs"].items()
            if run_objective == objective
        ]

        target = normalize(saved_targets[0])

        # Every seed should have used the same target.
        for other in saved_targets[1:]:
            np.testing.assert_allclose(
                normalize(other), target,
                rtol=1e-5, atol=1e-8,
            )

        targets[objective] = restrict(
            target, system_data["safe_mask"],
        )

    target_checks = pd.DataFrame([
        {
            "objective": objective,
            "grid_points": len(q),
            "probability_sum": q.sum(),
            "maximum_probability": q.max(),
        }
        for objective, q in targets.items()
    ]).set_index("objective")

    system_data.update({
        "targets": targets,
        "target_checks": target_checks,
    })

    print(f"System: {system}")
    display(target_checks)

In [ ]:
# Match each system's testing densities to its rollout scoring window.

MODELING_EXPERIMENT = "submission_nov"
MODELING_MODEL = "gru"

for system in SYSTEMS:
    system_data = analysis[system]
    grid = system_data["grid"]
    kde_params = system_data["reference_metadata"]["params"]["alg_params"]

    print(f"System: {system}")

    model_entries = list_modeling_results(
        system_name=system,
        model_type=MODELING_MODEL,
        exp_name=MODELING_EXPERIMENT,
        excitation_objective=list(OBJECTIVES),
        status="completed",
    )

    if not model_entries:
        raise ValueError(f"{system}: no completed modeling runs found.")

    sequence_lengths = {
        int(
            json.loads((entry["path"] / "experiment.json").read_text())
            ["params"]["model_trainer_params"]["sequence_length"]
        )
        for entry in model_entries
    }

    if len(sequence_lengths) != 1:
        raise ValueError(
            f"{system}: expected one training sequence length, "
            f"found {sorted(sequence_lengths)}."
        )

    sequence_length = sequence_lengths.pop()
    rollout_horizon = sequence_length - 1

    if rollout_horizon < 1:
        raise ValueError(f"{system}: rollout must contain at least one transition.")

    print(f"Training sequence: {sequence_length} states")
    print(f"Scored rollout: {rollout_horizon} future states")

    testing = load_testing_sequences(system)
    if not testing:
        raise ValueError(f"{system}: no testing datasets found.")

    families = tuple(sorted(testing))
    test_pmfs = {}
    test_checks = []

    for family in families:
        observations = np.asarray(testing[family]["observations"])
        actions = np.asarray(testing[family]["actions"])
        h = rollout_horizon

        if observations.shape[1] < h + 1 or actions.shape[1] < h:
            raise ValueError(
                f"{system}, {family}: trajectory is shorter than the scoring window."
            )

        observations = observations[:, :h + 1, :]
        actions = actions[:, :h, :]

        if not np.isfinite(observations).all() or not np.isfinite(actions).all():
            raise ValueError(f"{system}, {family}: nonfinite scoring-window data.")

        if system_data["use_actions"]:
            # Input pairs driving the scored transitions.
            features = np.concatenate(
                [observations[:, :-1, :], actions], axis=-1,
            )
        else:
            # Only future states included in the rollout score.
            features = observations[:, 1:, :]

        samples = features.reshape(1, -1, features.shape[-1])

        estimate = DensityEstimate.from_dataset(
            data_points=jnp.asarray(
                samples, dtype=jnp.float64 if use_f64 else jnp.float32,
            ),
            points_per_dim=kde_params["points_per_dim"],
            z_min=-kde_params["grid_extend"],
            z_max=kde_params["grid_extend"],
            bandwidth=system_data["bandwidth"],
            grid_batch_size=8,
        )

        np.testing.assert_allclose(estimate.z_g, grid, rtol=0, atol=2e-7)
        test_pmfs[family] = restrict(estimate.p, system_data["safe_mask"])

        test_checks.append({
            "family": family,
            "trajectories": features.shape[0],
            "steps_per_trajectory": h,
            "samples": samples.shape[1],
            "probability_sum": test_pmfs[family].sum(),
        })

    test_checks = pd.DataFrame(test_checks).set_index("family")

    system_data.update({
        "model_entries": model_entries,
        "sequence_length": sequence_length,
        "rollout_horizon": rollout_horizon,
        "families": families,
        "test_pmfs": test_pmfs,
        "test_checks": test_checks,
    })

    display(test_checks)

In [ ]:
for system in SYSTEMS:
    print(system)
    
    for key, arr in analysis[system]["densities"].items():
        print(key, ":", arr.shape)
    
    for key, arr in analysis[system]["targets"].items():
        print(key, ":", arr.shape)
    
    for key, arr in analysis[system]["test_pmfs"].items():
        print(key, ":", arr.shape)

    print()

In [ ]:
from itertools import combinations
from dmpe.utils.metrics import JSDLoss


def jsd(p, q):
    return float(
        JSDLoss(
            jnp.asarray(p)[:, None],
            jnp.asarray(q)[:, None],
        )
    )


for system in SYSTEMS:
    system_data = analysis[system]
    densities = system_data["densities"]

    # Compare distinct datasets; exclude comparisons with themselves.
    data_pairs = pd.DataFrame([
        {
            "objective_a": objective_a,
            "seed_a": seed_a,
            "objective_b": objective_b,
            "seed_b": seed_b,
            "jsd": jsd(p, q),
        }
        for ((objective_a, seed_a), p), ((objective_b, seed_b), q)
        in combinations(densities.items(), 2)
    ])

    # Mirror cross-objective pairs because JSD is symmetric.
    mirrored_pairs = (
        data_pairs.loc[
            data_pairs["objective_a"].ne(data_pairs["objective_b"])
        ]
        .rename(columns={
            "objective_a": "objective_b",
            "objective_b": "objective_a",
            "seed_a": "seed_b",
            "seed_b": "seed_a",
        })
    )

    data_data_summary = (
        pd.concat([data_pairs, mirrored_pairs], ignore_index=True)
        .groupby(["objective_a", "objective_b"])["jsd"]
        .agg(mean="mean", std="std", n_pairs="size")
    )

    DD = (
        data_data_summary["mean"]
        .unstack("objective_b")
        .reindex(index=OBJECTIVES, columns=OBJECTIVES)
    )

    system_data.update({
        "data_pairs": data_pairs,
        "data_data_summary": data_data_summary,
        "DD": DD,
    })

    print(f"System: {system}")
    display(DD.round(6))

In [ ]:
for system in SYSTEMS:
    system_data = analysis[system]
    targets = system_data["targets"]

    TT = pd.DataFrame(
        index=OBJECTIVES,
        columns=OBJECTIVES,
        dtype=float,
    )

    for objective_a in OBJECTIVES:
        for objective_b in OBJECTIVES:
            TT.loc[objective_a, objective_b] = jsd(
                targets[objective_a],
                targets[objective_b],
            )

    TT.index.name = "target_a"
    TT.columns.name = "target_b"

    system_data["TT"] = TT

    print(f"System: {system}")
    display(TT.round(6))

In [ ]:
for system in SYSTEMS:
    system_data = analysis[system]
    densities = system_data["densities"]
    targets = system_data["targets"]
    rows = []

    for (source_objective, seed), p in densities.items():
        for target_objective in OBJECTIVES:
            rows.append({
                "source_objective": source_objective,
                "seed": seed,
                "target_objective": target_objective,
                "jsd": jsd(p, targets[target_objective]),
            })

    data_target_scores = pd.DataFrame(rows)

    data_target_summary = (
        data_target_scores
        .groupby(["source_objective", "target_objective"])["jsd"]
        .agg(mean="mean", std="std", n_runs="size")
    )

    DT = (
        data_target_summary["mean"]
        .unstack("target_objective")
        .reindex(index=OBJECTIVES, columns=OBJECTIVES)
    )

    DT.index.name = "generating_objective"
    DT.columns.name = "scoring_target"

    system_data.update({
        "data_target_scores": data_target_scores,
        "data_target_summary": data_target_summary,
        "DT": DT,
    })

    print(f"System: {system}")
    display(DT.round(6))

In [ ]:
for system in SYSTEMS:
    system_data = analysis[system]
    densities = system_data["densities"]
    test_pmfs = system_data["test_pmfs"]
    families = system_data["families"]
    rows = []

    for (source_objective, seed), p in densities.items():
        for family in families:
            rows.append({
                "source_objective": source_objective,
                "seed": seed,
                "testing_family": family,
                "jsd": jsd(p, test_pmfs[family]),
            })

    data_test_scores = pd.DataFrame(rows)

    data_test_summary = (
        data_test_scores
        .groupby(["source_objective", "testing_family"])["jsd"]
        .agg(mean="mean", std="std", n_runs="size")
    )

    DTest = (
        data_test_summary["mean"]
        .unstack("testing_family")
        .reindex(index=OBJECTIVES, columns=families)
    )

    DTest.index.name = "generating_objective"
    DTest.columns.name = "testing_family"

    system_data.update({
        "data_test_scores": data_test_scores,
        "data_test_summary": data_test_summary,
        "DTest": DTest,
    })

    print(f"System: {system}")
    print(f"Testing horizon: {system_data['rollout_horizon']} future states")
    display(DTest.round(6))

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt


def plot_jsd_tables(DD, TT, DT, DTest, system, annotate=True):
    short = {
        "dmpe_uniform": "U",
        "dmpe_static_dynamics": "D",
        "dmpe_static_mixture_dynamics": "MD",
        "dmpe_static_curv": "C",
        "dmpe_static_mixture_curv": "MC",
        "dmpe_static_LOLA_ls": "L",
        "aprbs": "A",
        "multisine": "MS",
        "ocp": "O",
        "pi": "PI",
    }
    system_labels = {
        "pendulum": "Pendulum",
        "cart_pole": "Cart-pole",
        "fluid_tank": "Fluid tank",
        "pmsm_nominal": "PMSM",
    }
    panels = [
        ("Data vs data", DD),
        ("Target vs target", TT),
        ("Data vs target", DT),
        ("Data vs test", DTest),
    ]

    vmax = max(float(matrix.to_numpy().max()) for _, matrix in panels)
    norm = mpl.colors.Normalize(vmin=0, vmax=max(vmax, 1e-12))

    with plt.rc_context({
        "font.size": 7, "axes.titlesize": 7,
        "xtick.labelsize": 6, "ytick.labelsize": 7,
    }):
        fig, axes = plt.subplots(
            1, 4, figsize=(figurewidth_in, 2.2),
            sharey=True, layout="constrained",
            gridspec_kw={
                "width_ratios": [matrix.shape[1] for _, matrix in panels]
            },
        )

        for i, (ax, (title, matrix)) in enumerate(zip(axes, panels)):
            values = matrix.to_numpy()
            im = ax.imshow(
                values, cmap="viridis", norm=norm, aspect="equal",
            )
            ax.set_title(title, pad=4)
            ax.set_xticks(
                range(len(matrix.columns)),
                [short.get(c, c) for c in matrix.columns],
            )
            ax.set_yticks(
                range(len(matrix.index)),
                [short.get(r, r) for r in matrix.index],
            )
            ax.tick_params(length=0, labelleft=(i == 0))

            if annotate:
                for (row, col), value in np.ndenumerate(values):
                    label = (
                        f"{value:.0e}".replace("e-0", "e-")
                        if 0 < value < 1e-3 else f"{value:.2g}"
                    )
                    if label.startswith("0."):
                        label = label[1:]

                    ax.text(
                        col, row, label,
                        ha="center", va="center", fontsize=5.5,
                        color="white" if value < 0.5*vmax else "black",
                    )

        axes[0].set_ylabel("Objective")
        fig.suptitle(system_labels.get(system, system), fontsize=9)
        fig.colorbar(
            im, ax=axes.tolist(), label="JSD [nats]",
            fraction=0.02, pad=0.015, shrink=0.85,
        )

    return fig, axes

In [ ]:
figures = {}

for system in SYSTEMS:
    system_data = analysis[system]

    fig, axes = plot_jsd_tables(
        system_data["DD"],
        system_data["TT"],
        system_data["DT"],
        system_data["DTest"],
        system=system,
    )

    figures[system] = fig

    # Optional export:
    # fig.savefig(
    #     f"paper_fig/excitation_jsd_{system}.pdf",
    #     bbox_inches="tight",
    # )

    plt.show()

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path


def plot_jsd_overview(
    analysis, systems, objectives, width=6.0, height=6.6, annotate=True,
):
    comparisons = [
        ("DD", r"$\mathrm{JSD}(\hat{\bm{p}}_x, \hat{\bm{p}}_y)$"),
        ("TT", r"$\mathrm{JSD}(\bm{p}^*_x, \bm{p}^*_y)$"),
        ("DT", r"$\mathrm{JSD}(\bm{p}^*_x, \hat{\bm{p}}_y)$"),
        ("DTest", r"$\mathrm{JSD}(\bm{p}^{\mathrm{test}}_x,\hat{\bm{p}}_y)$"),
    ]
    objective_labels = {
        "dmpe_uniform": r"$\mathrm{U}$",
        "dmpe_static_dynamics": r"$\mathrm{dyn}$",
        "dmpe_static_mixture_dynamics": r"$\mathrm{mix,dyn}$",
        "dmpe_static_curv": r"$\mathrm{curv}$",
        "dmpe_static_mixture_curv": r"$\mathrm{mix,curv}$",
        "dmpe_static_LOLA_ls": r"$\mathrm{LOLA}$",
    }
    family_labels = {
        "aprbs": r"$\mathrm{APRBS}$",
        "multisine": r"$\mathrm{MS}$",
        "ocp":  r"$\mathrm{OCP}$",
        "pi":  r"$\mathrm{PI}$",
    }
    system_labels = {
        "fluid_tank": "Fluid tank",
        "pendulum": "Pendulum",
        "cart_pole": "Cart-pole",
        "pmsm_nominal": "PMSM",
    }
    max_families = max(analysis[s]["DTest"].shape[1] for s in systems)

    with plt.rc_context({
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }):
        fig, axes = plt.subplots(
            len(systems), 4,
            figsize=(width, height),
            squeeze=False,
            layout="constrained",
            gridspec_kw={
                "width_ratios": [len(objectives)] * 3 + [max_families],
            },
        )
        for ax in axes[:, :3].ravel()[1:]:
            ax.sharex(axes[0, 0])

        for row, system in enumerate(systems):
            # One shared scale for the four comparisons of this system.
            vmax = max(
                float(analysis[system][key].to_numpy().max())
                for key, _ in comparisons
            )
            norm = mpl.colors.Normalize(vmin=0, vmax=max(vmax, 1e-12))

            for col, (key, title) in enumerate(comparisons):
                ax = axes[row, col]
                matrix = analysis[system][key].reindex(index=objectives)
                if key != "DTest":
                    matrix = matrix.reindex(columns=objectives)

                values = matrix.to_numpy()
                im = ax.imshow(
                    values, cmap="viridis", norm=norm, aspect="equal",
                )
                ax.set_anchor("W")
                if key == "DTest":
                    # Reserve equal column space even for a single test family.
                    ax.set_xlim(-0.5, max_families - 0.5)

                ax.set_yticks(
                    range(len(objectives)),
                    [objective_labels.get(o, o) for o in objectives],
                )
                ax.tick_params(axis="y", labelleft=(col == 0))

                labels = family_labels if key == "DTest" else objective_labels
                
                ax.set_xticks(
                    range(len(matrix.columns)),
                    [labels.get(c, c) for c in matrix.columns],
                )
                ax.tick_params(
                    axis="x",
                    labelbottom=(row in (2, 3)),
                    labelrotation=90,
                )
                ax.tick_params(length=0, pad=2)

                for spine in ax.spines.values():
                    spine.set_visible(False)

                if annotate:
                    for (i, j), value in np.ndenumerate(values):
                        label = (
                            f"{value:.0e}".replace("e-0", "e-")
                            if 0 < value < 1e-3 else f"{value:.2g}"
                        )
                        if label.startswith("0."):
                            label = label[1:]

                        ax.text(
                            j, i, label,
                            ha="center", va="center", fontsize=5,
                            color="white" if norm(value) < 0.5 else "black",
                        )

                if row == 0:
                    ax.set_title(title, pad=7)
                if col == 0:
                    ax.set_ylabel(system_labels.get(system, system), labelpad=9)

            colorbar = fig.colorbar(
                im, ax=axes[row, :].tolist(),
                orientation="vertical", shrink=0.9,
                fraction=0.02, pad=0.025, aspect=30, panchor=False,
            )
            colorbar.set_label("JSD", fontsize=7, labelpad=4)
            colorbar.ax.tick_params(labelsize=6, length=2, pad=2)

    return fig, axes


fig_4x4, axes_4x4 = plot_jsd_overview(
    analysis, SYSTEMS, OBJECTIVES,
    width=figurewidth_in, height=6.6, annotate=True,
)

output_dir = Path("paper_fig") / "submission_nov_curv64"
output_dir.mkdir(exist_ok=True)

# Keep the exported page at the chosen dimensions.
with plt.rc_context({"savefig.bbox": None, "pdf.fonttype": 42}):
    fig_4x4.savefig(output_dir / "excitation_jsd_all_systems_4x4.pdf")
    fig_4x4.savefig(output_dir / "excitation_jsd_all_systems_4x4.png", dpi=300)

plt.show()

In [ ]:
def plot_jsd_overview_reduced(
    analysis, systems, objectives, width=6.0, height=6.6, annotate=True,
):
    comparisons = [
        ("TT", r"$\mathrm{JSD}(\bm{p}^*_x, \bm{p}^*_y)$"),
        ("DT", r"$\mathrm{JSD}(\bm{p}^*_x, \hat{\bm{p}}_y)$"),
        ("DD", r"$\mathrm{JSD}(\hat{\bm{p}}_x, \hat{\bm{p}}_y)$"),
    ]
    objective_labels = {
        "dmpe_uniform": r"$\mathrm{U}$",
        "dmpe_static_dynamics": r"$\mathrm{dyn}$",
        "dmpe_static_mixture_dynamics": r"$\mathrm{mix,dyn}$",
        "dmpe_static_curv": r"$\mathrm{curv}$",
        "dmpe_static_mixture_curv": r"$\mathrm{mix,curv}$",
        "dmpe_static_LOLA_ls": r"$\mathrm{LOLA}$",
    }
    system_labels = {
        "fluid_tank": "Fluid tank",
        "pendulum": "Pendulum",
        "cart_pole": "Cart-pole",
        "pmsm_nominal": "PMSM",
    }

    with plt.rc_context({
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }):
        fig, axes = plt.subplots(
            len(systems), len(comparisons),
            figsize=(width, height),
            squeeze=False,
            sharex=True,
            layout="constrained",
        )

        for row, system in enumerate(systems):
            matrices = [
                analysis[system][key].reindex(
                    index=objectives, columns=objectives,
                )
                for key, _ in comparisons
            ]

            # Shared scale across the three retained comparisons.
            vmax = max(float(matrix.to_numpy().max()) for matrix in matrices)
            norm = mpl.colors.Normalize(vmin=0, vmax=max(vmax, 1e-12))

            for col, ((key, title), matrix) in enumerate(
                zip(comparisons, matrices)
            ):
                ax = axes[row, col]
                values = matrix.to_numpy()
                im = ax.imshow(
                    values, cmap="viridis", norm=norm, aspect="equal",
                )
                ax.set_anchor("W")

                labels = [objective_labels.get(o, o) for o in objectives]
                ax.set_xticks(range(len(objectives)), labels)
                ax.set_yticks(range(len(objectives)), labels)
                ax.tick_params(axis="y", labelleft=(col == 0))
                ax.tick_params(
                    axis="x",
                    labelbottom=(row >= len(systems) - 1),
                    labelrotation=90,
                )
                ax.tick_params(length=0, pad=2)

                for spine in ax.spines.values():
                    spine.set_visible(False)

                if annotate:
                    for (i, j), value in np.ndenumerate(values):
                        label = (
                            f"{value:.0e}".replace("e-0", "e-")
                            if 0 < value < 1e-3 else f"{value:.2g}"
                        )
                        if label.startswith("0."):
                            label = label[1:]

                        ax.text(
                            j, i, label,
                            ha="center", va="center", fontsize=7,
                            color="white" if norm(value) < 0.5 else "black",
                        )

                if row == 0:
                    ax.set_title(title, pad=7)
                if col == 0:
                    ax.set_ylabel(
                        system_labels.get(system, system), labelpad=9,
                    )

            colorbar = fig.colorbar(
                im, ax=axes[row, :].tolist(),
                orientation="vertical",
                shrink=0.9,
                fraction=0.04,
                pad=0.025,
                aspect=14,
                panchor=False,
            )
            colorbar.set_label("JSD", fontsize=8, labelpad=5)
            colorbar.ax.tick_params(labelsize=8, length=3, pad=3)

    return fig, axes


fig_4x3, axes_4x3 = plot_jsd_overview_reduced(
    analysis, SYSTEMS, OBJECTIVES,
    width=figurewidth_in, height=6.6, annotate=True,
)

output_dir = Path("paper_fig") / "submission_nov_curv64"
output_dir.mkdir(parents=True, exist_ok=True)

with plt.rc_context({"savefig.bbox": None, "pdf.fonttype": 42}):
    fig_4x3.savefig(output_dir / "excitation_jsd_all_systems_4x3.pdf")
    fig_4x3.savefig(
        output_dir / "excitation_jsd_all_systems_4x3.png", dpi=300,
    )

plt.show()

In [ ]:
raise

In [ ]:
for system in SYSTEMS:
    system_data = analysis[system]
    targets = system_data["targets"]
    test_pmfs = system_data["test_pmfs"]
    families = system_data["families"]
    rows = []

    for source_objective, p in targets.items():
        for family in families:
            rows.append({
                "source_objective": source_objective,
                "seed": seed,
                "testing_family": family,
                "jsd": jsd(p, test_pmfs[family]),
            })

    data_test_scores = pd.DataFrame(rows)

    data_test_summary = (
        data_test_scores
        .groupby(["source_objective", "testing_family"])["jsd"]
        .agg(mean="mean", std="std", n_runs="size")
    )

    DTest = (
        data_test_summary["mean"]
        .unstack("testing_family")
        .reindex(index=OBJECTIVES, columns=families)
    )

    DTest.index.name = "generating_objective"
    DTest.columns.name = "testing_family"

    print(f"System: {system}")
    print(f"Testing horizon: {system_data['rollout_horizon']} future states")
    display(DTest.round(6))